# Two-pass GLCast on real CTIS aggregate signals

This small historical **API demonstration is not a paper benchmark or a new
validation cohort**. It uses only the bundled three-state aggregate subset.
Attribution, transformations, source hashes and reuse terms are recorded in
`examples/ctis_smoke/provenance.json` and its accompanying README.

We inspect the actual observations, fit the uniform first pass, derive adaptive
group weights, refit with those weights, and independently calculate losses.
The configuration deliberately differs from the paper to keep execution small.
All displayed results below come from executed code, not illustrative data.


In [1]:
from pathlib import Path
from dataclasses import replace
import hashlib
import importlib.metadata as metadata
import json
import os
import time
import numpy as np
import pandas as pd
from IPython.display import display
import glcast_method
from glcast_method import (
    MethodConfig, load_long_csv, fit_first_pass, adaptive_weights, run_walkforward,
)

started = time.perf_counter()
here = Path.cwd()
roots = [here, here.parent, here.parent/"glcast_github", here.parent.parent]
example = next((root/"examples/ctis_smoke" for root in roots
                if (root/"examples/ctis_smoke/ctis_aggregate.csv").exists()), None)
assert example is not None, "Run from the method repository or its notebooks directory"
installed = "site-packages" in Path(glcast_method.__file__).parts
if os.environ.get("GLCAST_REQUIRE_WHEEL_IMPORT") == "1":
    assert installed, "Clean validation must use the installed distribution"
display(pd.DataFrame([
    {"package": name, "version": metadata.version(name)}
    for name in ["glcast-method", "numpy", "pandas", "scipy", "celer"]
]))
print("Import mode:", "installed distribution" if installed else "source checkout")


,package,version
0,glcast-method,0.1.0
1,numpy,2.4.6
2,pandas,3.0.6
3,scipy,1.17.1
4,celer,0.7.4


Import mode: installed distribution


## 1. Provenance and input integrity

In [2]:
provenance = json.loads((example/"provenance.json").read_text())
data_path = example/"ctis_aggregate.csv"
input_sha256 = hashlib.sha256(data_path.read_bytes()).hexdigest()
assert input_sha256 == provenance["subset_sha256"]
print("Input SHA-256:", input_sha256)
print("Attribution:", provenance["attribution"])
print("Documented license:", provenance["license"])
display(pd.DataFrame(provenance["sources"])[
    ["signal", "filename", "selected_rows", "missing_value", "sha256"]])


Input SHA-256: 0bb21da8310b85c08c26e8962013c0f6c14c9b4284f0a2c7e727e8b7ab9565ec
Attribution: U.S. COVID-19 Trends and Impact Survey (CTIS), Delphi Group at Carnegie Mellon University, in partnership with Facebook. Aggregate signals selected and reshaped for this example; no endorsement implied.
Documented license: CC BY 4.0


,signal,filename,selected_rows,missing_value,sha256
0,raw_wcli,covidcast-fb-survey-raw_wcli-2020-04-06-to-202...,270,0,c6c2a7977c5a17e47108a4801aa1f608e9bad98a6f9131...
1,raw_whh_cmnty_cli,covidcast-fb-survey-raw_whh_cmnty_cli-2020-04-...,270,0,790f77cf2e2c4beff5e8abc5277c4971396e8dadee21ae...
2,smoothed_wtested_14d,covidcast-fb-survey-smoothed_wtested_14d-2020-...,270,0,86fb20bce629e5ed3856056d89017a83f29a88bb966899...
3,smoothed_wtested_positive_14d,covidcast-fb-survey-smoothed_wtested_positive_...,270,0,0ec7beae6a45903dccc0eff503c2fb5988a7c0418c8628...


In [3]:
observations = pd.read_csv(data_path, parse_dates=["date"])
assert len(observations) == 270
assert not observations.duplicated(["state", "date"]).any()
signals = [provenance["target"], *provenance["auxiliaries"]]
assert np.isfinite(observations[signals].to_numpy()).all()
qc = observations.groupby("state").agg(
    rows=("date", "size"), first_date=("date", "min"),
    last_date=("date", "max"),
)
assert qc.rows.eq(90).all()
display(qc)
display(observations.head(9))
display(observations[signals].describe().T[["count", "mean", "std", "min", "max"]])


,rows,first_date,last_date
state,,,
CA,90,2021-01-01,2021-03-31
NY,90,2021-01-01,2021-03-31
TX,90,2021-01-01,2021-03-31


,state,date,raw_wcli,raw_whh_cmnty_cli,smoothed_wtested_14d,smoothed_wtested_positive_14d
0,CA,2021-01-01,1.660790,31.063652,18.333859,18.491533
1,NY,2021-01-01,0.932375,30.803064,18.194230,12.002862
2,TX,2021-01-01,1.488114,35.349379,14.775190,25.313782
3,CA,2021-01-02,1.477957,30.734409,18.206095,18.701721
4,NY,2021-01-02,1.131670,27.241643,17.851427,12.505389
5,TX,2021-01-02,1.485172,34.111206,14.789238,25.886233
6,CA,2021-01-03,1.141830,29.745047,18.113062,19.346122
7,NY,2021-01-03,0.759512,25.476159,17.580754,12.978517
8,TX,2021-01-03,1.336248,35.141535,14.657611,25.420064


,count,mean,std,min,max
raw_wcli,270.0,0.962451,0.358442,0.321398,2.111562
raw_whh_cmnty_cli,270.0,22.643540,7.356411,11.211272,39.036763
smoothed_wtested_14d,270.0,14.585865,3.478688,7.819264,19.630886
smoothed_wtested_positive_14d,270.0,14.475422,6.309725,5.184999,28.205124


## 2. Explicit non-paper configuration

The input subset has no missing signal values and is not imputed or standardized
by this example's data-selection step. Internal model fitting retains the public
implementation's numerical recipe. The short windows, three-alpha grid and
daily serving below are demonstration choices, not the paper settings.


In [4]:
cfg = MethodConfig.from_json(example/"config.json")
target, features = load_long_csv(
    data_path, target=provenance["target"],
    auxiliaries=provenance["auxiliaries"], config=cfg,
)
assert target.shape == (90, 3) and len(features) == 3
display(pd.DataFrame([
    {"setting": key, "value": json.dumps(value)}
    for key, value in cfg.to_dict().items()
]))


,setting,value
0,backend,"""classic"""
1,unit,"""D"""
2,train_window,28
3,val_window,7
4,test_window,7
5,retrain_every,7
6,feature_lags,"[0, 1]"
7,ar_lags,"[1, 2]"
8,alphas,"[0.001, 0.01, 0.1]"
9,alpha_y_scale,false


## 3. Uniform first pass and retrospective adaptive weights

In [5]:
first = fit_first_pass(target, features, cfg)
assert len(first["fits"]) == 576
assert not first["fits"].duplicated(
    ["origin", "state", "alpha", "feature"]).any()
display(first["alpha_picks"])
display(first["fits"].groupby(["origin", "state"]).size()
        .rename("coefficient_rows").reset_index())
group_weights = adaptive_weights(
    first["fits"], [*features, "AR"], alpha_picks=first["alpha_picks"],
)
assert np.isclose(np.mean(list(group_weights.values())), 1.)
assert all(np.isfinite(value) and value > 0 for value in group_weights.values())
display(pd.Series(group_weights, name="adaptive_group_weight")
        .rename_axis("group").to_frame())


,block,alpha,origin
0,1,NaN,2021-01-28
1,2,0.1,2021-02-04
2,3,0.1,2021-02-11
3,4,0.1,2021-02-18
4,5,0.1,2021-02-25
5,6,0.1,2021-03-04
6,7,0.1,2021-03-11
7,8,0.1,2021-03-18


,origin,state,coefficient_rows
0,2021-01-28,CA,24
1,2021-01-28,NY,24
2,2021-01-28,TX,24
3,2021-02-04,CA,24
4,2021-02-04,NY,24
5,2021-02-04,TX,24
6,2021-02-11,CA,24
7,2021-02-11,NY,24
8,2021-02-11,TX,24
9,2021-02-18,CA,24


,adaptive_group_weight
group,
raw_whh_cmnty_cli,1.121034
smoothed_wtested_14d,0.923785
smoothed_wtested_positive_14d,1.206937
AR,0.748244


## 4. Weighted final pass and served predictions

The adaptive weights summarize the full historical first pass. The earlier-origin
penalty selection also uses the implementation's test-block history. This is
therefore a retrospective demonstration, not release-aware prospective validation.


In [6]:
adaptive = replace(cfg, group_weights=group_weights)
result = run_walkforward(target, features, adaptive)
predictions = result.predictions.copy()
assert len(result.alpha_picks) == 8
assert len(predictions) == 147 and len(result.weights) == 63
assert not predictions.duplicated(["state", "time_value"]).any()
assert np.isfinite(predictions[["y_true", "y_pred"]].to_numpy()).all()
assert predictions.time_value.min() == pd.Timestamp("2021-02-05")
assert predictions.time_value.max() == pd.Timestamp("2021-03-25")
display(result.alpha_picks)
display(predictions.head(12))
display(predictions.groupby("state").agg(
    scored_cells=("time_value", "size"), first=("time_value", "min"),
    last=("time_value", "max")))


,block,origin,alpha
0,1,2021-01-28,NaN
1,2,2021-02-04,0.1
2,3,2021-02-11,0.1
3,4,2021-02-18,0.1
4,5,2021-02-25,0.1
5,6,2021-03-04,0.1
6,7,2021-03-11,0.1
7,8,2021-03-18,0.1


,state,time_value,y_true,y_pred,origin,block
0,CA,2021-02-05,0.954986,0.811276,2021-02-04,2
1,CA,2021-02-06,0.685958,0.746387,2021-02-04,2
2,CA,2021-02-07,0.827828,0.703751,2021-02-04,2
3,CA,2021-02-08,0.711055,0.660883,2021-02-04,2
4,CA,2021-02-09,0.796345,0.663907,2021-02-04,2
5,CA,2021-02-10,0.533152,0.575611,2021-02-04,2
6,CA,2021-02-11,0.741515,0.529162,2021-02-04,2
7,CA,2021-02-12,0.977446,0.617798,2021-02-11,3
8,CA,2021-02-13,1.238518,0.651939,2021-02-11,3
9,CA,2021-02-14,1.125247,0.719469,2021-02-11,3


,scored_cells,first,last
state,,,
CA,49,2021-02-05,2021-03-25
NY,49,2021-02-05,2021-03-25
TX,49,2021-02-05,2021-03-25


## 5. Independent saved-prediction losses

In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

score_rows = []
for state, part in predictions.groupby("state"):
    y = part.y_true.to_numpy(float)
    p = np.maximum(part.y_pred.to_numpy(float), 0.)
    error = p-y
    centered = y-y.mean()
    rmse = float(np.sqrt(np.mean(error**2)))
    mae = float(np.mean(np.abs(error)))
    r2 = float(1-(error@error)/(centered@centered))
    np.testing.assert_allclose(rmse, np.sqrt(mean_squared_error(y,p)), atol=1e-12)
    np.testing.assert_allclose(mae, mean_absolute_error(y,p), atol=1e-12)
    np.testing.assert_allclose(r2, r2_score(y,p), atol=1e-12)
    score_rows.append({"state":state,"n":len(y),"RMSE":rmse,"MAE":mae,"R2":r2})
scores = pd.DataFrame(score_rows)
display(scores)
display(scores[["RMSE","MAE","R2"]].mean().rename("equal_state_mean").to_frame())


,state,n,RMSE,MAE,R2
0,CA,49,0.189411,0.147954,-0.145975
1,NY,49,0.251476,0.201049,-0.007048
2,TX,49,0.280015,0.220975,-0.222406


,equal_state_mean
RMSE,0.240301
MAE,0.189993
R2,-0.125143


## 6. Raw weights versus display shares

Raw combination weights are nonnegative but need not sum to one. Normalization
below is for display only and does not change the predictions. A zero-total row
would remain undefined rather than be assigned a fabricated composition.


In [8]:
weights = result.weights.copy()
assert np.isfinite(weights.weight).all() and weights.weight.ge(0).all()
keys = ["state","origin"]
weights["raw_total"] = weights.groupby(keys).weight.transform("sum")
weights["display_share"] = weights.weight/weights.raw_total.replace(0,np.nan)
zero_totals = weights.loc[weights.raw_total.eq(0),keys].drop_duplicates()
nonzero = weights.loc[weights.raw_total.gt(0)]
np.testing.assert_allclose(nonzero.groupby(keys).display_share.sum(),1.,atol=1e-12)
display(weights.head(12))
display(weights.groupby("state").raw_total.agg(["min","median","max"]))
print("Undefined zero-total compositions:",len(zero_totals))


,block,origin,state,candidate,weight,gamma,w_stack,w_qagg,raw_total,display_share
0,2,2021-02-04,CA,self,9.641146e-01,0.411765,0.953993,9.711996e-01,0.981057,9.827303e-01
1,2,2021-02-04,CA,TX,6.260988e-07,0.411765,0.000002,2.705538e-190,0.981057,6.381879e-07
2,2,2021-02-04,CA,NY,1.694196e-02,0.411765,0.000001,2.880036e-02,0.981057,1.726909e-02
3,2,2021-02-04,NY,self,9.692623e-01,0.411765,0.925351,1.000000e+00,0.969267,9.999950e-01
4,2,2021-02-04,NY,TX,4.882066e-06,0.411765,0.000012,0.000000e+00,0.969267,5.036863e-06
5,2,2021-02-04,NY,CA,3.608065e-13,0.411765,0.000000,6.133710e-13,0.969267,3.722467e-13
6,2,2021-02-04,TX,self,9.949548e-01,0.411765,0.987747,1.000000e+00,0.994955,1.000000e+00
7,2,2021-02-04,TX,CA,2.903596e-44,0.411765,0.000000,4.936114e-44,0.994955,2.918320e-44
8,2,2021-02-04,TX,NY,2.156607e-09,0.411765,0.000000,3.666232e-09,0.994955,2.167543e-09
9,3,2021-02-11,CA,self,9.709879e-01,0.411765,0.960161,9.785668e-01,0.983596,9.871820e-01


,min,median,max
state,,,
CA,0.978204,0.983596,1.034581
NY,0.968872,1.006884,1.040597
TX,0.927398,0.994216,1.027043


Undefined zero-total compositions: 0


## 7. Save reproducible numerical outputs

No data plot is needed for this small installation check. The substantive
prediction, score, alpha-history and weight tables remain available for reuse.


In [9]:
output = Path("ctis_notebook_outputs")
output.mkdir(exist_ok=True)
predictions.to_csv(output/"predictions.csv",index=False)
scores.to_csv(output/"per_state_metrics.csv",index=False)
weights.to_csv(output/"raw_and_display_weights.csv",index=False)
first["alpha_picks"].to_csv(output/"first_pass_alpha_picks.csv",index=False)
result.alpha_picks.to_csv(output/"final_alpha_picks.csv",index=False)
receipt = {
    "status":"complete","purpose":"non-paper historical permitted-data demonstration",
    "input_sha256":input_sha256,"observations":270,"prediction_rows":len(predictions),
    "weight_rows":len(weights),"scheduled_origins":len(result.alpha_picks),
    "adaptive_group_weights":group_weights,"equal_state_metrics":
        scores[["RMSE","MAE","R2"]].mean().to_dict(),
    "installed_distribution":installed,"elapsed_seconds":time.perf_counter()-started,
    "prospective_validation":False,
}
(output/"receipt.json").write_text(json.dumps(receipt,indent=2)+"\n")
display(pd.DataFrame([{
    "file":file.name,"bytes":file.stat().st_size,
    "sha256":hashlib.sha256(file.read_bytes()).hexdigest()
} for file in sorted(output.iterdir()) if file.is_file()]))


,file,bytes,sha256
0,final_alpha_picks.csv,152,10ca5f134f9cd2697ff3dcd553804d5fa1130768bc827b...
1,first_pass_alpha_picks.csv,152,935d0e7c963d6ea992e3249ee9f1d46fda493cf42049a6...
2,per_state_metrics.csv,218,c199f8b2d90389dadc5f0e11af11a2bf972748f8ff23c7...
3,predictions.csv,8235,3dcdd3cb81801642a2c78381b90382ec125ed60ea19956...
4,raw_and_display_weights.csv,8476,047c3d1716366bd7df046c1070abb7b29ea0f921b08d61...
5,receipt.json,721,5095f5a34efce720e7fbaf05afdd7ca567f7e6991e6ad2...


## Scope and limitations

- These are three jurisdictions and a short historical period, with no
  independent cohort, prospective evaluation, baseline tournament or paper-scale
  performance claim.
- Adaptive first-pass weighting and whole-test-block selection are retrospective.
  Calibration/training overlap and target/feature complete-case eligibility are
  retained from the public implementation.
- Agreement among formulas on the same saved predictions is computational
  validation, not independent scientific replication.
- A successful notebook validates this exercised environment and example.
  It does not guarantee bitwise equality on every platform or a no-celer path.
